In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import nltk
import emoji
import spacy
import string
import unicodedata
import datetime
import random
from sklearn.utils import shuffle
from sklearn.feature_extraction.text import CountVectorizer

portuguese_stopwords = nltk.corpus.stopwords.words('portuguese')

In [9]:
df = pd.read_csv('df_regex.csv', sep=';', dtype={'COMMENT_TEXT': str})

df = df.sample(10000)

In [10]:
def preprocess_data(data, 
                    columns,
                    null = True):
    
    df = data[columns]
    
    if null:
        df = df.dropna()
    
    return df


def preprocess_text(text, 
                    remove_stop = True, 
                    stem_words = False, 
                    remove_mentions_hashtags = True
                   ):
    """
    eg:
    input: preprocess_text("@water #dream hi hello where are you going be there tomorrow happening happen happens",  
    stem_words = True) 
    output: ['tomorrow', 'happen', 'go', 'hello']
    """

    # Remove emojis
    emoji_pattern = re.compile("[" "\U0001F1E0-\U0001F6FF" "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r"", text)
    text = "".join([x for x in text if x not in emoji.UNICODE_EMOJI])
    
    # corrects the bug that eliminates letters from words with accents
    text = ''.join(ch for ch in unicodedata.normalize('NFKD', text) 
    if not unicodedata.combining(ch))

    # removes special characters 
    if remove_mentions_hashtags:
        text = re.sub(r"@(\w+)", " ", text)
        text = re.sub(r"#(\w+)", " ", text)

    #
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    regex = re.compile('[' + re.escape(string.punctuation) + '0-9\\r\\t\\n]')
    nopunct = regex.sub(" ", text.lower())
    words = (''.join(nopunct)).split()
    
    # removes stopwords with less than two letters
    if(remove_stop):
        words = [w for w in words if w not in portuguese_stopwords]
        words = [w for w in words if len(w) > 2]  

    if(stem_words):
        stemmer = PorterStemmer()
        words = [stemmer.stem(w) for w in words]

    return list(words)


def tokenizacao(df):
    
    # gets the number of rows and columns from the dataframe
    rows, cols = df.shape

    # creates the dataset column with the vectorized text
    df['token'] = [preprocess_text(df["COMMENT_TEXT"][row]) for row in range(rows)]
    
    return df

#######################################################################################################################
def classifica_label(df):
    
    # faz o drop dos na do dataframe de entrada
    df = df.dropna()
    
    # reseta o index do datraframe de entrada
    df2 = df.reset_index().drop('index',axis=1)
    
    # tokeniza os comentários do dataframe de entrada
    df2 = tokenizacao(df2)
    
    # faz uma cópia do dataframe de entrada
    df3 = df2.copy() 
    
    # reseta o index da cópia do dataframe
    df3 = df3.reset_index()
    
    # renomeia o index do dataframe cópia para usar o index_1 como a key do comentário
    df3 = df3.rename(columns={'index':'index_1'})

    # cria uma lista vazia com os comentários positivos
    lista_positivos = []

    # procura as frases que contém palavras positivas
    for n, objeto in enumerate(positivos_token):
        lista_1 = []
        for i in range(len(df2)):
            if (objeto[0] in df2.token[i]):
                lista_1.append([df2.token[i], df2.index[i]])
        for n, item in enumerate(lista_positivos):
            lista_1 = [item1 for item1 in lista_1 if item1 not in item]
        lista_positivos.append(lista_1)

    # cria uma lista vazia com os comentários negativos
    lista_negativos = []

    # procura as frases que contém palavras negativas
    for n, objeto in enumerate(negativos_token):
        lista_2 = []
        for i in range(len(df2)):
            if (objeto[0] in df2.token[i]):
                lista_2.append([df2.token[i], df2.index[i]])
        for n, item in enumerate(lista_negativos):
            lista_2 = [item1 for item1 in lista_2 if item1 not in item]
        lista_negativos.append(lista_2)
     
    # cria uma nova lista vazia com palavras positivas 
    lista_positivos_nova = []

    # Cria nova lista com palavras positivas usando o extend de listas
    for n, objeto in enumerate(lista_positivos):
        lista_temp = [item[0] for item in objeto] 
        lista_positivos_nova.extend(lista_temp)

    # cria uma nova lista vazia com palavras negativas
    lista_negativos_nova = []

    # Cria nova lista com palavras negativas usando o extend de listas
    for n, objeto in enumerate(lista_negativos):
        lista_temp = [item[0] for item in objeto] 
        lista_negativos_nova.extend(lista_temp)
    
    # cria uma lista vazia com palavras neutras
    lista_neutro = []
    
    # se a palavra não estiver nem na lista positiva nem na lista negativa, vira palavra neutra
    for i in range(len(df2)):
        if (df2.token[i] not in lista_positivos_nova) and (df2.token[i] not in lista_negativos_nova):
            lista_neutro.append([df2.token[i], df2.index[i]])
    
    # cria uma lista vazia de labels
    labels = []

    # gera o label positivo para comentários com palavras positivas
    for n, item in enumerate(lista_positivos):
        for i, objeto in enumerate(item):
            labels.append(2)

    # gera o label negativo para comentários com palavras negativas
    for n, item in enumerate(lista_negativos):
        for i, objeto in enumerate(item):
            labels.append(1)

    # gera o label neutro para comentários com palavras neutras
    for n, item in enumerate(lista_neutro):
            labels.append(0)
     
    # cria um novo dataframe com apenas uma linha, com 2 colunas (frase tokenizada e índice)
    # para sentimentos positivos
    df_positivo = pd.DataFrame(lista_positivos[0])
    
    # concatena o resto da lista para sentimentos positivos
    for n, objeto in enumerate(lista_positivos[1:]):
        df_positivo = pd.concat([df_positivo,pd.DataFrame(objeto)])

    # renomeia as colunas do dataframe criado para sentimentos positivos
    df_positivo = df_positivo.rename(columns=({0:'token',1:'index_1'})) 

    # cria um novo dataframe com apenas uma linha, com 2 colunas (frase tokenizada e índice)
    # para sentimentos negativos
    df_negativo = pd.DataFrame(lista_negativos[0])
    
    # concatena o resto da lista para sentimentos negativos
    for n, objeto in enumerate(lista_negativos[1:]):
        df_negativo = pd.concat([df_negativo,pd.DataFrame(objeto)])
    
    # renomeia as colunas do dataframe criado para sentimentos negativos
    df_negativo = df_negativo.rename(columns={0:'token',1:'index_1'})

    # cria um novo dataframe com apenas uma linha, com 2 colunas (frase tokenizada e índice)
    # para sentimentos neutros
    df_neutro = pd.DataFrame(lista_neutro)
    
    # renomeia as colunas do dataframe criado para sentimentos neutros
    df_neutro = df_neutro.rename(columns={0:'token',1:'index_1'})

    # concatena os dataframes positivos e negativos
    df_pos_neg = pd.concat([df_positivo,df_negativo])
    
    # concatena com o dataframe neutro
    df_pos_neg_neutro = pd.concat([df_pos_neg,df_neutro])

    # cria a coluna com os labels definidos anteriormente
    df_pos_neg_neutro['labels'] = labels

    # Agrupa comentários classificados com ambiguidade (possivelmente ironias, e.g gostei muito... horrível)
    # df_pos_neg_neutro = df_pos_neg_neutro.groupby('index_1').agg(token=('token','first'),
    #                             labels=('labels','first')).reset_index()
    
    # colunas para deixar no df3
    columns_keep = ['index_1',
                    'AP2',
                    'COUNTRY',
                    'COMMENT_TEXT',
                    #'token'
                   ]
    
    # faz o merge entre os dataframes 
    df_final = pd.merge(df_pos_neg_neutro,
                        df3[columns_keep], 
                        how = 'left',
                        left_on = 'index_1',
                        right_on = 'index_1'
                       )
    
    # colunas para deixar no dataframe df_final
    columns_keep_final = ['AP2',
                          'COUNTRY',
                          'COMMENT_TEXT',
                          'labels',
                          'index_1',
                          'token'
                         ]
    
    # renomeia o nome da coluna labels para sentimento_labels
    df_final = df_final[columns_keep_final].rename(columns={'labels': 'sentimento_labels'})
    
    # aplica o dicionário de sentimentos
    df_final.sentimento_labels = df_final.sentimento_labels.map(dic_sentimentos) 
    
    return df_final

In [11]:
palavras_positivas = ['bom',
                      'lindo',
                      'fantastico',
                      'top',
                      'maravilha',
                      'amou',
                      'amando',
                      'melhor',
                      'parabéns',
                      'elogio',
                      'incrível',
                      'adorei',
                      'excelente',
                      'certinho',
                      'perfeito',
                      'gostei muito'
                     ]

palavras_negativas = ['fragil',
                      'insatisfeito',
                      'horrível',
                      'pessimo',
                      'lixo',
                      'bosta',
                      'devagar'
                     ]

dic_sentimentos = {
    2:'Positivo',
    1:'Negativo',
    0:'Neutro'
}

#
positivos_token = [preprocess_text(item) for item in palavras_positivas]

#
negativos_token = [preprocess_text(item) for item in palavras_negativas]

In [13]:
#pd.set_option("max_rows", None)
pd.options.display.max_colwidth = 200

df_final = classifica_label(df)

df_final

,AP2,COUNTRY,COMMENT_TEXT,sentimento_labels,index_1,token
0,SEDA,BR,Bom Custo benefício,Positivo,1,"[bom, custo, beneficio]"
1,SEDA,BR,Gostei muito muito bom mesmo obrigado,Positivo,4,"[gostei, bom, obrigado]"
2,SEDA,BR,"Para o meu proposito, o produto esta bom.\nAtende minhas necessidades.",Positivo,6,"[proposito, produto, bom, natende, necessidades]"
3,SEDA,BR,"O produto é bom. Contudo, barulhento. A borracha da porta do freezer veio amassada, e o letreiro veio torto.",Positivo,9,"[produto, bom, contudo, barulhento, borracha, porta, freezer, veio, amassada, letreiro, veio, torto]"
4,SEDA,BR,"Bom demais, rápido,Câmara muito boa, recomendo a todos",Positivo,12,"[bom, demais, rapido, camara, boa, recomendo, todos]"
...,...,...,...,...,...,...
10016,SEDA,BR,"Logo no dia que chegou apresentou problemas, pedi troca e até hoje espero vir a nova máquina e nada de chegar já tem mais de 30 dias",Neutro,9992,"[logo, dia, chegou, apresentou, problemas, pedi, troca, ate, hoje, espero, vir, nova, maquina, nada, chegar, dias]"
10017,SEDA,BR,Eu já conhecia o produto por isso comprei,Neutro,9994,"[conhecia, produto, comprei]"
10018,SEDA,BR,"Superou todas minha expectativas, recomendo este produto.",Neutro,9996,"[superou, todas, expectativas, recomendo, produto]"
10019,SEDA,BR,"Chegou dentro do prazo, atende bem as expectativas",Neutro,9998,"[chegou, dentro, prazo, atende, bem, expectativas]"
